# Imports

In [2]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [3]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm.notebook as tqdm
import transformers

c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: True


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [5]:
# class Tokenizer(transformers.GPT2Tokenizer):

#     def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
#         if token_ids_1 is None:
#             return [self.bos_token_id, *token_ids_0, self.eos_token_id]

#         return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]

In [6]:
tokenizer: transformers.PreTrainedTokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

In [7]:
assert isinstance(tokenizer.eos_token_id, int)
pad_token_id: int = tokenizer.eos_token_id + 1

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [8]:
df = pd.read_csv("../../data/rnn_seq2seq/rus-eng.tsv", sep="\t", usecols=[1, 3], names=["ru", "en"])

In [9]:
df

,ru,en
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?
...,...,...
830313,Меня не поставили в известность.,I was not informed.
830314,Меня не поставили в известность.,I have not been informed.
830315,Она не поставила меня в известность.,She didn't inform me.
830316,Мне просто надо было увидеть это самому.,I just had to see this for myself.


In [10]:
df["ru"].str.len().describe()

count    830318.000000
mean         30.799881
std          15.759152
min           2.000000
25%          22.000000
50%          28.000000
75%          36.000000
max         841.000000
Name: ru, dtype: float64

In [11]:
df["en"].str.len().describe()

count    830318.000000
mean         31.370901
std          15.583791
min           3.000000
25%          23.000000
50%          29.000000
75%          37.000000
max        1127.000000
Name: en, dtype: float64

In [12]:
import pickle

In [ ]:
TOKENIZE_NOW = False

In [14]:
if TOKENIZE_NOW:
    tokens_ru = [tokenizer.encode(text) for text in df["ru"]]
    with open("../../data/rnn_seq2seq/ru.pkl", "wb") as f:
        pickle.dump(tokens_ru, f, protocol=pickle.HIGHEST_PROTOCOL)

In [15]:
if TOKENIZE_NOW:
    tokens_en = [tokenizer.encode(text) for text in df["en"]]
    with open("../../data/rnn_seq2seq/en.pkl", "wb") as f:
            pickle.dump(tokens_en, f, protocol=pickle.HIGHEST_PROTOCOL)

In [16]:
with open("../../data/rnn_seq2seq/ru.pkl", "rb") as f:
    tokens_ru = pickle.load(f)

with open("../../data/rnn_seq2seq/en.pkl", "rb") as f:
    tokens_en = pickle.load(f)

In [17]:
class Seq2SeqDataset(keras.utils.PyDataset):
    def __init__(
        self,
        input_sequences: list[list[int]],
        output_sequences: list[list[int]],
        tokenizer: transformers.PreTrainedTokenizer,
        pad_token_id: int,
        batch_size: int = 128,
        shuffle: bool = True,
        rng: np.random.Generator | None = None,
        **kwargs,
    ):
        super().__init__(**kwargs)

        if rng is None:
            rng = np.random.default_rng()

        self.input = input_sequences
        self.output = output_sequences
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = rng
        self.tokenizer = tokenizer
        self.pad_token_id = pad_token_id
        self.indices = np.arange(len(input_sequences))

        self.on_epoch_end()

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_indices = self.indices[start:end]

        encoder_input_tokens = [self.input[i] for i in batch_indices]
        decoder_input_tokens = [
            [tokenizer.bos_token_id] + self.output[i] for i in batch_indices
        ]
        decoder_output_tokens = [
            self.output[i] + [tokenizer.eos_token_id] for i in batch_indices
        ]

        encoder_input = keras.utils.pad_sequences(
            encoder_input_tokens,
            padding="post",
            value=pad_token_id,
        )
        decoder_input = keras.utils.pad_sequences(
            decoder_input_tokens,
            padding="post",
            value=pad_token_id,
        )
        decoder_output = keras.utils.pad_sequences(
            decoder_output_tokens,
            padding="post",
            value=pad_token_id,
        )

        mask = (decoder_output != pad_token_id).astype(np.float32)

        return (encoder_input, decoder_input), decoder_output, mask

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [18]:
ds = Seq2SeqDataset(tokens_ru, tokens_en, tokenizer, pad_token_id)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [ ]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
) -> keras.Model:
    '''Creates a model with RNN architecture for sequence to sequence classification.

    Arguments:
        units: dimensionality of RNN cells
        n_tokens: number of tokens in the tokenizer dictionary
        n_labels: number of labels to be predicted
        n_stacks: number of RNN cells in the stack (1 -- no stacking)
        bidirectional: whether or not the model is bidirectional
        name: the model name
        cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

    Returns:
        The model'''
    ...

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [ ]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    model: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text`translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''
    ...